# Retrieval-Augmented Generation (RAG) Course

This notebook consolidates all the RAG labs: Document Chunking, Vector Search using embeddings, and End-to-End RAG using SentenceTransformers + GPT-2.


## Topic 1: Document Chunking (تقسيم المستندات)


In [ ]:
# Step 1) تعريف النص وتقسيمه / Define raw text and chunk_text function
document = """
Retrieval-Augmented Generation (RAG) is a technique that grants LLMs access to external data.
RAG combines retrieval of documents with generation of text.
First, we load the knowledge source and split it into smaller text chunks.
Chunking is necessary because models have limited context windows.
We usually use overlapping chunks to ensure semantic continuity between borders.
"""

def chunk_text(text, chunk_size=100, overlap=20):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += (chunk_size - overlap)
    return chunks


In [ ]:
# Step 2) تجربة التقسيم بمقاييس مختلفة / Test chunking function
chunks = chunk_text(document, chunk_size=120, overlap=30)
print(f"Total Chunks Generated: {len(chunks)}")
print("="*50)
for i, c in enumerate(chunks):
    print(f"Chunk {i+1}:\n{c.strip()}\n")


## Topic 2: Vector Search (البحث الدلالي الجيبي)


In [ ]:
# Step 1) استيراد المكتبات وتوليد المتجهات / Import embedding library and generate vectors
# %pip install sentence-transformers -q
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer("all-MiniLM-L6-v2")
chunks = [
    "Retrieval-Augmented Generation (RAG) grants LLMs access to external data.",
    "RAG combines document retrieval with text generation.",
    "Chunking splits text into smaller pieces because of context limits.",
    "Overlapping chunks ensure semantic continuity between borders."
]

chunk_embeddings = model.encode(chunks)
print("Embedding Matrix Shape:", chunk_embeddings.shape)


In [ ]:
# Step 2) حساب درجة التشابه الجيبي / Calculate Cosine Similarity manual function
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

query = "How does RAG access external databases?"
query_embedding = model.encode(query)

scores = [cosine_similarity(query_embedding, emb) for emb in chunk_embeddings]
print(f"Query: '{query}'")


In [ ]:
# Step 3) ترتيب وتصفية نتائج البحث / Rank search results
ranked_indices = np.argsort(scores)[::-1]
print("Ranked Search Results:")
print("="*50)
for idx in ranked_indices:
    print(f"Score: {scores[idx]:.4f} | Chunk: '{chunks[idx]}'")


## Topic 3: End-to-End RAG Pipeline (خط إنتاج الـ RAG الكامل)


In [ ]:
# Step 1) استدعاء النماذج وتجهيز قاعدة المعرفة / Load models and setup knowledge base
from sentence_transformers import SentenceTransformer
from transformers import pipeline
import numpy as np

embed_model = SentenceTransformer("all-MiniLM-L6-v2")
generator = pipeline("text-generation", model="gpt2")

knowledge_base = [
    "The capital of France is Paris. It is known for Eiffel Tower.",
    "The capital of Japan is Tokyo. It is famous for its sushi and technology.",
    "The capital of Australia is Canberra. It was selected as a compromise between Sydney and Melbourne."
]
kb_embeddings = embed_model.encode(knowledge_base)


In [ ]:
# Step 2) بناء دالة الاسترجاع الدلالي / Define retrieval function
def retrieve(query):
    q_emb = embed_model.encode(query)
    scores = [np.dot(q_emb, kb_emb) / (np.linalg.norm(q_emb) * np.linalg.norm(kb_emb)) for kb_emb in kb_embeddings]
    best_idx = np.argmax(scores)
    return knowledge_base[best_idx]


In [ ]:
# Step 3) دمج السياق وتوليد الإجابة / RAG Execution
query = "What is the capital of Japan and what is it famous for?"
context = retrieve(query)

# Construct Prompt
prompt = f"Answer the query based on the context.\nContext: {context}\nQuery: {query}\nAnswer:"

# Generate text
output = generator(prompt, max_new_tokens=20, pad_token_id=50256)
print("RAG Prompt:")
print(prompt)
print("\nGenerated Response:")
print(output[0]['generated_text'])
